# 🛠️ Llama Hata Kurtarma Notebook

Bu notebook, Llama 3.1 8B base testindeki 185 JSON parse hatasını kurtarır ve sonuç dosyasını günceller.

**Akış:**
1. Checkpoint dosyasından ham yanıtları yükle
2. 4 farklı kurtarma stratejisi uygula (markdown stripping, literal newline fix, partial JSON, regex extraction)
3. Kurtarılan satırları checkpoint'e geri yaz
4. Tam skorlama (`run_evaluation` + `calculate_soft_accuracy`) yeniden çalıştır
5. Güncel sonuç dosyasını kaydet

**Önkoşul:** `acik_kaynak_test_FINAL.ipynb` notebook'undaki A.1 → A.5 ve B.1 → B.4 hücreleri çalıştırılmış olmalı (yani `df`, `emb_model`, `nli_pipe`, `run_evaluation` vb. hazır olmalı).

**Süre:** ~5 dakika (model yükleme yok, sadece text parsing + skorlama)

---

## 1. Checkpoint'i yükle ve hata istatistiği

In [ ]:
import os
print("Drive mount durumu:")
print("  /content/drive var mı?", os.path.exists("/content/drive"))
print("  MyDrive var mı?", os.path.exists("/content/drive/MyDrive"))
print("  BitirmeProjesi var mı?", os.path.exists("/content/drive/MyDrive/BitirmeProjesi"))
print("  checkpoints klasörü var mı?", os.path.exists("/content/drive/MyDrive/BitirmeProjesi/checkpoints"))

Drive mount durumu:
  /content/drive var mı? True
  MyDrive var mı? True
  BitirmeProjesi var mı? True
  checkpoints klasörü var mı? True


In [ ]:
import json
import re
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
CHECKPOINT_FILE = f"{PROJECT_FOLDER}/checkpoints/generation_llama31_8b_base_20260514.json"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"

assert os.path.exists(CHECKPOINT_FILE), f"Checkpoint bulunamadı: {CHECKPOINT_FILE}"

with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
    results = json.load(f)

errors = [r for r in results if r.get('error') is not None]
successes = [r for r in results if r.get('error') is None]

print(f"Toplam satır: {len(results)}")
print(f"Başarılı: {len(successes)}")
print(f"Hatalı: {len(errors)}")
print(f"Hata oranı: {len(errors)/len(results)*100:.1f}%")

Mounted at /content/drive
Toplam satır: 488
Başarılı: 303
Hatalı: 185
Hata oranı: 37.9%


## 2. Kurtarma fonksiyonları (4 strateji)

Sırayla denenir, ilk başarılı olan kullanılır:
1. **Markdown stripping** — ` ```json ... ``` ` bloğunu temizle
2. **Literal newline fix** — String içindeki kontrol karakterlerini escape et
3. **Concatenated strings** — Hata 2 tipi: reasoning ayrı string'lere bölünmüş, birleştir
4. **Regex extraction** — JSON yarım kalmışsa reasoning + answer + confidence'ı ayrı ayrı çekmeye çalış

In [ ]:
def strategy_1_markdown(raw: str):
    """```json ... ``` veya ``` ... ``` bloğunu temizle, sonra parse et."""
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            return None
    return None


def strategy_2_literal_newline(raw: str):
    """
    Hata 3 tipi: JSON içinde literal \n var, escape edilmemiş.
    Çözüm: string değerlerinin içindeki literal newline/tab/return karakterlerini escape et.
    """
    # En dıştaki { ... } bloğunu bul
    s = raw.strip()
    if not s.startswith('{'):
        # JSON'un başlangıcını bul
        idx = s.find('{')
        if idx == -1:
            return None
        s = s[idx:]

    # Brace matching ile JSON'un sonunu bul
    depth = 0
    in_string = False
    escape = False
    end_idx = -1
    for i, ch in enumerate(s):
        if escape:
            escape = False
            continue
        if ch == '\\':
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end_idx = i
                break

    if end_idx == -1:
        return None

    candidate = s[:end_idx+1]

    # String içindeki literal kontrol karakterlerini escape et
    # Yaklaşım: string olduğunu bildiğimiz bölgelerde \n, \r, \t karakterlerini escape et
    result_chars = []
    in_string = False
    escape = False
    for ch in candidate:
        if escape:
            result_chars.append(ch)
            escape = False
            continue
        if ch == '\\':
            result_chars.append(ch)
            escape = True
            continue
        if ch == '"':
            in_string = not in_string
            result_chars.append(ch)
            continue
        if in_string and ch in '\n\r\t':
            # Literal kontrol karakterini escape et
            if ch == '\n':
                result_chars.append('\\n')
            elif ch == '\r':
                result_chars.append('\\r')
            elif ch == '\t':
                result_chars.append('\\t')
        else:
            result_chars.append(ch)

    fixed = ''.join(result_chars)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        return None


def strategy_3_concatenated_strings(raw: str):
    """
    Hata 2 tipi: reasoning bir tek string değil, ayrı string'lere bölünmüş.
    Örnek: "reasoning": "...", "1. Şu...", "2. Bu...", "answer": ...
    Çözüm: "reasoning" key'inden sonraki 'orphan' string'leri birleştir.
    """
    s = raw.strip()
    if not s.startswith('{'):
        idx = s.find('{')
        if idx == -1:
            return None
        s = s[idx:]

    # "reasoning" değerini bul, sonrasındaki ayrı string'leri yakalamaya çalış
    # Regex: tüm "...": "..." çiftlerini bul (basit)
    # Bu strateji daha çok kurtarmaya yöneliktir, perfect parse değil

    reasoning_match = re.search(r'"reasoning"\s*:\s*"([^"]*)"', s, re.DOTALL)
    answer_match = re.search(r'"answer"\s*:\s*(?:"([^"]*)"|([-+]?\d+(?:\.\d+)?)|(true|false|null))', s)
    confidence_match = re.search(r'"confidence"\s*:\s*(\d+(?:\.\d+)?)', s)

    if reasoning_match is None:
        return None

    reasoning = reasoning_match.group(1)

    # "reasoning"den sonra, "answer"dan önceki yetim string'leri bul ve reasoning'e ekle
    end_of_reasoning = reasoning_match.end()
    if answer_match:
        start_of_answer = answer_match.start()
        middle = s[end_of_reasoning:start_of_answer]
    else:
        middle = s[end_of_reasoning:]

    # middle içindeki tüm "..." string'lerini topla
    orphan_strings = re.findall(r'"([^"]*)"', middle)
    # Sayısal/null gibi şeyleri zaten bypass etmiş oluyoruz, sadece tırnak içindekiler
    extra = ' '.join(orphan_strings)
    if extra:
        reasoning = f"{reasoning} {extra}".strip()

    # Answer
    if answer_match:
        ans = answer_match.group(1) or answer_match.group(2) or answer_match.group(3)
    else:
        ans = None

    # Confidence
    if confidence_match:
        try:
            conf = float(confidence_match.group(1))
        except ValueError:
            conf = None
    else:
        conf = None

    # Minimum: reasoning + answer olmalı; answer yoksa kurtaramayız
    if ans is None:
        return None

    return {
        'reasoning': reasoning,
        'answer': ans,
        'confidence': conf if conf is not None else 50,   # confidence yoksa default 50
    }


def strategy_4_regex_extraction(raw: str):
    """
    Son çare: JSON yarım kalmış, ama reasoning + answer + confidence regex ile çekilebilir.
    Truncated yanıtlar için.
    """
    # reasoning: tırnaklar arası, çok satırlı kabul
    reasoning_match = re.search(
        r'"reasoning"\s*:\s*"((?:[^"\\]|\\.)*?)"',
        raw, re.DOTALL
    )
    answer_match = re.search(
        r'"answer"\s*:\s*(?:"((?:[^"\\]|\\.)*?)"|([-+]?\d+(?:\.\d+)?))',
        raw, re.DOTALL
    )
    confidence_match = re.search(r'"confidence"\s*:\s*(\d+(?:\.\d+)?)', raw)

    if reasoning_match is None or answer_match is None:
        return None

    reasoning = reasoning_match.group(1)
    ans = answer_match.group(1) or answer_match.group(2)

    if confidence_match:
        try:
            conf = float(confidence_match.group(1))
        except ValueError:
            conf = 50
    else:
        conf = 50   # default, çünkü yarım kalmış

    return {
        'reasoning': reasoning,
        'answer': ans,
        'confidence': conf,
    }


def try_all_strategies(raw: str):
    """4 stratejiyi sırayla dene, ilk başarılı olanı dön."""
    for strategy_name, strategy in [
        ('markdown', strategy_1_markdown),
        ('literal_newline', strategy_2_literal_newline),
        ('concatenated', strategy_3_concatenated_strings),
        ('regex_extraction', strategy_4_regex_extraction),
    ]:
        parsed = strategy(raw)
        if parsed is not None and isinstance(parsed, dict):
            # Minimum validation
            if 'reasoning' in parsed and 'answer' in parsed:
                return parsed, strategy_name
    return None, None


print("✓ Kurtarma fonksiyonları tanımlandı")

✓ Kurtarma fonksiyonları tanımlandı


## 3. Hatalı satırları kurtarmayı dene

In [ ]:
recovered = []
still_failed = []
strategy_counts = {'markdown': 0, 'literal_newline': 0, 'concatenated': 0, 'regex_extraction': 0}

for r in errors:
    raw = r.get('model_raw_response', '')
    if not raw:
        still_failed.append(r)
        continue

    parsed, strategy = try_all_strategies(raw)
    if parsed is None:
        still_failed.append(r)
        continue

    strategy_counts[strategy] += 1
    recovered.append({
        'question_id': r['question_id'],
        'model_raw_response': raw,
        'model_reasoning': parsed.get('reasoning'),
        'model_answer': parsed.get('answer'),
        'model_confidence': parsed.get('confidence'),
        'error': None,
        'recovery_strategy': strategy,
    })

print(f"📊 Kurtarma Sonuçları")
print(f"   Hatalı satır sayısı: {len(errors)}")
print(f"   Kurtarılan: {len(recovered)} ({len(recovered)/len(errors)*100:.1f}%)")
print(f"   Hâlâ başarısız: {len(still_failed)}")
print(f"\n   Strateji dağılımı:")
for strat, cnt in strategy_counts.items():
    print(f"     {strat}: {cnt}")

print(f"\n   Toplam başarılı (eskiler + kurtarılanlar): {len(successes) + len(recovered)} / {len(results)}")
print(f"   Yeni başarı oranı: {(len(successes) + len(recovered))/len(results)*100:.1f}%")

📊 Kurtarma Sonuçları
   Hatalı satır sayısı: 185
   Kurtarılan: 149 (80.5%)
   Hâlâ başarısız: 36

   Strateji dağılımı:
     markdown: 0
     literal_newline: 32
     concatenated: 117
     regex_extraction: 0

   Toplam başarılı (eskiler + kurtarılanlar): 452 / 488
   Yeni başarı oranı: 92.6%


## 4. Kurtarılan örnekleri kontrol et

In [ ]:
# Her stratejiden 1 örnek göster, beklenen alanları doğrula
print("Kurtarılan örnekler (her stratejiden 1 tane):\n")
seen_strategies = set()
for r in recovered:
    s = r['recovery_strategy']
    if s in seen_strategies:
        continue
    seen_strategies.add(s)
    print(f"--- Strateji: {s} | question_id: {r['question_id']} ---")
    print(f"  Reasoning (ilk 150 char): {str(r['model_reasoning'])[:150]}")
    print(f"  Answer: {r['model_answer']}")
    print(f"  Confidence: {r['model_confidence']}")
    print()

# Hâlâ başarısız olanlardan 2 örnek (raw response başlangıçları)
print("\nHâlâ kurtarılamayan örnekler (ilk 2):")
for r in still_failed[:2]:
    print(f"--- question_id: {r['question_id']} ---")
    raw = r.get('model_raw_response', '')
    print(f"  Raw (ilk 300 char): {raw[:300]}")
    print(f"  Raw uzunluğu: {len(raw)} karakter")
    print()

Kurtarılan örnekler (her stratejiden 1 tane):

--- Strateji: concatenated | question_id: NR-2-003 ---
  Reasoning (ilk 150 char): Adım adım düşünce süreci: 1. Depodan 2 koli sevkiyat yapıldığında sistemdeki kod 'P' değerine düşmektedir. Bu, P'nin 10 ile çarpıldığında 2 koli sevki
  Answer: 2
  Confidence: 100.0

--- Strateji: literal_newline | question_id: NR-1-005 ---
  Reasoning (ilk 150 char): Adım 1: Polat, Kenan, Ramiz ve Sevda'nın yaşlarının aritmetik ortalaması bulmak için, önce Yiğit ve Ömer'in yaşlarını çıkaracağız. 
    Adım 2: Yiğit 
  Answer: 12.5
  Confidence: 100


Hâlâ kurtarılamayan örnekler (ilk 2):
--- question_id: NR-1-001 ---
  Raw (ilk 300 char): {
    "reasoning": "Adım 1: C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından farklı olması gerektiği anlaşılıyor. Bu, C sayısının yüzler basamağını 4 artırarak elde edilen sonucun D sayısından 4 daha büyük olması gerektiği anlamına gelir.\nAdım 2: C sayısının yüzler basamağ
  Raw uzunluğu: 2767 ka

## 5. Checkpoint'i güncelle (kurtarılanları başarılı olarak işaretle)

In [ ]:
# Önce yedek al
import shutil
from datetime import datetime

backup_file = CHECKPOINT_FILE.replace('.json', f'_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json')
shutil.copy(CHECKPOINT_FILE, backup_file)
print(f"✓ Yedek alındı: {backup_file}")

# Yeni results listesi: orijinal başarılılar + kurtarılanlar + hâlâ başarısızlar
# question_id'ye göre dict yap, sonra orijinal sırayı koru
by_qid = {}
for r in successes:
    by_qid[r['question_id']] = r
for r in recovered:
    # recovery_strategy alanını çıkar (orijinal şemaya uy)
    clean = {k: v for k, v in r.items() if k != 'recovery_strategy'}
    by_qid[r['question_id']] = clean
for r in still_failed:
    by_qid[r['question_id']] = r

# Orijinal sırayı koru
updated_results = [by_qid[r['question_id']] for r in results]

with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
    json.dump(updated_results, f, ensure_ascii=False, indent=2)

# Doğrulama
valid_count = sum(1 for r in updated_results if r.get('error') is None)
print(f"✓ Checkpoint güncellendi: {CHECKPOINT_FILE}")
print(f"  Yeni başarılı satır sayısı: {valid_count} / {len(updated_results)}")

✓ Yedek alındı: /content/drive/MyDrive/BitirmeProjesi/checkpoints/generation_llama31_8b_base_20260514_backup_20260514_195745.json
✓ Checkpoint güncellendi: /content/drive/MyDrive/BitirmeProjesi/checkpoints/generation_llama31_8b_base_20260514.json
  Yeni başarılı satır sayısı: 452 / 488


## 6. Tam skorlamayı yeniden çalıştır (run_evaluation + soft accuracy)

**Önkoşul:** Ana notebook'tan A.1 → A.5 + B.1 → B.4 hücrelerini çalıştırmış olman gerekir. Aşağıdaki hücre `df`, `run_evaluation`, `calculate_soft_accuracy`, `RESULTS_PATH`, `TODAY` değişkenlerinin tanımlı olduğunu varsayar.

In [ ]:
# ============================================================
# Skorlama için ana notebook'un gerekli kısmını kur
# ============================================================
import sys, os, shutil
import pandas as pd
import torch
from datetime import datetime

PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
sys.path.insert(0, PROJECT_FOLDER)

# Cache temizle
pycache_path = f"{PROJECT_FOLDER}/core/__pycache__"
if os.path.exists(pycache_path):
    shutil.rmtree(pycache_path)

from core.faithfulness_utils import *

# Veri seti
DATASET_TAG = "base"
DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Base_Questions 489.csv"
CHECKPOINT_PATH = f"{PROJECT_FOLDER}/checkpoints/"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"
TODAY = datetime.now().strftime('%Y%m%d')

# Config set
set_config(
    vector_w=0.4, nli_w=0.6,
    llm_model="local-open-source", temp=0, max_tok=2048,
    checkpoint_path=CHECKPOINT_PATH
)

# Veri yükle
df = load_and_preprocess_data(DATA_PATH)
print(f"✓ df yüklendi: {len(df)} satır")

# Embedding ve NLI modelleri yükle
from sentence_transformers import SentenceTransformer
from transformers import pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"► Embedding model yükleniyor (device={device})...")
emb_model = SentenceTransformer("intfloat/multilingual-e5-large", device=device)
print("► NLI pipeline yükleniyor...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=0 if device == "cuda" else -1
)
set_models(emb_model, nli_pipe, None, None, None)

print("\n✓ Skorlama için hazır")

✓ Veri yüklendi: 489 satır
  ⚠️ 1 satır eksik veri nedeniyle çıkarıldı
  Evidence parça sayıları: min=1, max=7, ortalama=2.8
✓ df yüklendi: 488 satır
► Embedding model yükleniyor (device=cuda)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]


✓ Skorlama için hazır


In [ ]:
import pandas as pd
from tqdm.notebook import tqdm

# Ana notebook'taki global değişkenleri kullan
MODEL_SHORT_NAME = "llama31_8b"
DATASET_TAG = "base"
MODEL_DISPLAY_NAME = "Llama 3.1 8B Instruct (recovered)"

# Güncellenmiş checkpoint'i tekrar oku ve df ile birleştir
with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
    gen_results = json.load(f)

gen_df = pd.DataFrame(gen_results)
merged = df.merge(gen_df, on='question_id', how='left')
print(f"✓ Generation checkpoint yüklendi (post-recovery): {len(merged)} satır")

# Geçerli satırları al
valid = merged[
    merged['error'].isna() &
    merged['model_reasoning'].notna() &
    (merged['model_reasoning'].astype(str).str.len() > 0)
].copy().reset_index(drop=True)
print(f"► Skorlanacak satır: {len(valid)} / {len(merged)}")

# 1) run_evaluation (faithfulness + accuracy + epistemic)
df_evaluated = run_evaluation(valid)

# 2) Soft accuracy
print("\n► Soft accuracy hesaplanıyor...")
soft_rows = []
for _, row in tqdm(df_evaluated.iterrows(), total=len(df_evaluated), desc="Soft Acc"):
    try:
        r = calculate_soft_accuracy(row['model_answer'], row['gold_answer'])
    except Exception as e:
        r = {'soft_match': False, 'accuracy_score': 0, 'match_type': f'error: {e}', 'semantic_sim': 0.0, 'nli_entailment': 0.0}
    soft_rows.append({
        'question_id': row['question_id'],
        'soft_match': r['soft_match'],
        'soft_accuracy_score': r['accuracy_score'],
        'match_type': r['match_type'],
        'answer_semantic_sim': r['semantic_sim'],
        'answer_nli_entailment': r['nli_entailment'],
    })
soft_df = pd.DataFrame(soft_rows)
df_evaluated = df_evaluated.merge(soft_df, on='question_id', how='left')

# Kaydet (yeni dosya adı, _recovered suffix)
output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{DATASET_TAG}_recovered_{TODAY}.csv"
df_evaluated.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Skorlama tamamlandı, kaydedildi: {output_path}")

✓ Generation checkpoint yüklendi (post-recovery): 488 satır
► Skorlanacak satır: 452 / 488
► Evaluation başlıyor...
  Toplam soru: 452


Evaluating:   0%|          | 0/452 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



✓ Evaluation tamamlandı!

► Soft accuracy hesaplanıyor...


Soft Acc:   0%|          | 0/452 [00:00<?, ?it/s]


✓ Skorlama tamamlandı, kaydedildi: /content/drive/MyDrive/BitirmeProjesi/results/evaluation_acik_llama31_8b_base_recovered_20260514.csv


## 7. Yeni özet (kurtarma sonrası)

In [ ]:
print("\n" + "=" * 70)
print(f"📊 ÖZET — Llama 3.1 8B Instruct (recovered)")
print("=" * 70)
print(f"Toplam soru: {len(df_evaluated)}")

exact_rate = df_evaluated['accuracy_score'].mean() * 100
print(f"\nExact Accuracy: {exact_rate:.1f}%  ({df_evaluated['accuracy_score'].sum()}/{len(df_evaluated)})")

soft_rate = df_evaluated['soft_accuracy_score'].mean() * 100
print(f"Soft Accuracy:  {soft_rate:.1f}%  ({df_evaluated['soft_accuracy_score'].sum()}/{len(df_evaluated)})")

print(f"\nOrtalama semantic_similarity: {df_evaluated['semantic_similarity'].mean():.3f}")
print(f"Ortalama nli_score:           {df_evaluated['nli_score'].mean():.3f}")
print(f"Ortalama faithfulness_score:  {df_evaluated['faithfulness_score'].mean():.3f}")
print(f"Ortalama evidence_recall:     {df_evaluated['evidence_recall'].mean():.3f}")

print(f"\nOrtalama confidence: {df_evaluated['confidence_normalized'].mean()*100:.1f}%")
print(f"Ortalama epistemic_gap: {df_evaluated['epistemic_gap'].mean():.3f}")

print("\nHallucination risk dağılımı:")
for risk, cnt in df_evaluated['hallucination_risk'].value_counts().items():
    print(f"   {risk}: {cnt} ({cnt/len(df_evaluated)*100:.1f}%)")

print("\nCalibration tipi dağılımı:")
for cal, cnt in df_evaluated['calibration_type'].value_counts().items():
    print(f"   {cal}: {cnt} ({cnt/len(df_evaluated)*100:.1f}%)")

print("=" * 70)


📊 ÖZET — Llama 3.1 8B Instruct (recovered)
Toplam soru: 452

Exact Accuracy: 28.1%  (127/452)
Soft Accuracy:  37.2%  (168/452)

Ortalama semantic_similarity: 0.874
Ortalama nli_score:           0.519
Ortalama faithfulness_score:  0.661
Ortalama evidence_recall:     1.000

Ortalama confidence: 97.4%
Ortalama epistemic_gap: 0.315

Hallucination risk dağılımı:
   ORTA: 306 (67.7%)
   DÜŞÜK: 129 (28.5%)
   YÜKSEK: 17 (3.8%)

Calibration tipi dağılımı:
   Overconfident: 424 (93.8%)
   İyi Kalibre: 27 (6.0%)
   Underconfident: 1 (0.2%)


## 8. Karşılaştırma: önceki sonuç (303 satır) vs şimdiki (kurtarma sonrası)

In [ ]:
# Eski sonuç dosyasını oku
old_path = f"{RESULTS_PATH}evaluation_acik_llama31_8b_base_20260514.csv"

if os.path.exists(old_path):
    old_df = pd.read_csv(old_path)

    def s(df, col):
        return df[col].mean() if col in df.columns else None

    print(f"{'Metrik':<30} {'Eski (303)':<15} {'Yeni (recovered)':<20}")
    print('-' * 65)
    metrics = [
        ('Toplam satır', None),
        ('Exact Accuracy (%)', 'accuracy_score'),
        ('Soft Accuracy (%)', 'soft_accuracy_score'),
        ('Faithfulness', 'faithfulness_score'),
        ('Semantic Similarity', 'semantic_similarity'),
        ('NLI Score', 'nli_score'),
        ('Confidence', 'confidence_normalized'),
        ('Epistemic Gap', 'epistemic_gap'),
    ]

    for label, col in metrics:
        if col is None:
            print(f"{label:<30} {len(old_df):<15} {len(df_evaluated):<20}")
        else:
            old_val = s(old_df, col)
            new_val = s(df_evaluated, col)
            if old_val is None or new_val is None:
                continue
            if 'Accuracy' in label or 'Confidence' in label:
                old_str = f"{old_val*100:.1f}%"
                new_str = f"{new_val*100:.1f}%"
            else:
                old_str = f"{old_val:.3f}"
                new_str = f"{new_val:.3f}"
            print(f"{label:<30} {old_str:<15} {new_str:<20}")
else:
    print(f"Eski sonuç dosyası bulunamadı: {old_path}")

Metrik                         Eski (303)      Yeni (recovered)    
-----------------------------------------------------------------
Toplam satır                   303             452                 
Exact Accuracy (%)             29.0%           28.1%               
Soft Accuracy (%)              40.3%           37.2%               
Faithfulness                   0.656           0.661               
Semantic Similarity            0.875           0.874               
NLI Score                      0.510           0.519               
Confidence                     97.4%           97.4%               
Epistemic Gap                  0.321           0.315               
